# R-tag pipeline — single run walkthrough

End-to-end **KB331** sample: filter GDS → per-resonator RTEG layouts with MBE/MTE routing.

**Docs:** [README](../README.md) · [CLAUDE](../CLAUDE.md) · run top-to-bottom from `python_code/`.

| Step | What it does | Main call |
|------|----------------|-----------|
| 1 | Validate inputs | layermap + file check |
| 2 | Find resonators / vias | `identify` |
| 3 | Nudge resonator clear of GSG pad metal (placement clearance) | `prep_resonator_ppd` |
| 4 | Place in die frame + attach preserved interconnect | `prep_rteg_in_frame`, `attach_preserved_filter_interconnect_all` |
| 5.1–5.2 | Collect geometry → classify signal/ground | `collect_geometry_roles`, `classify_nodes` |
| 5.2b | If pad Y outside collar span: align collar mouth to pad Y (+60 µm) for routing; re-prep | `signal_shift_for_resonator` → `shift_overrides` |
| 5.2c | `collar_extend` only: clip wild preserved MTE + MBE to body bbox + 20 µm; clean routes + cap | `normalize_mte_collar_extent_all` |
| 5.3 | Signal route (intercepts from die) + ground filler | `build_all_routes` |
| 5.3b | `collar_extend` only: carve MBE rectangle filler outside MTE-extension span | inside `build_all_routes` |
| 5.5 | Split rectangle filler: bridge + frame-ring cap | `apply_filler_bridge_all_routes` |
| 5.4 | Straighten MBE release-hole keepout arcs → 3 chords | `clean_all_routes` |
| export | Complete routed GDS | `export_route_new_gds` → `draft_output/route_outputs/` |

**KB331 split (8 resonators):** `center_pad` MTE signal → indices **1, 3, 4, 6** · `collar_extend` MBE signal → **0, 2, 5, 7**.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import gdstk
import pandas as pd
from IPython.display import display


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ROUTE_OUTPUTS = DRAFT / "route_outputs"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

## Define inputs

Set paths to the filter GDS, die frame, GSG probe template, and Skyworks layermap. All paths are under `input_files/`. The next code cell assigns `FILTER`, `FRAME`, `PPD`, `LAYERMAP`.


In [ ]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process inputs

**~30 s read**

Confirms all four input files exist and prints a short inventory (size + role). Also reads the frame template bbox so you know the probe window size before placement.

**Output:** sanity-check table ΓÇö aborts if anything is missing.


In [ ]:
# Ensure all referenced input files exist; abort on missing inputs

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS ΓÇö resonators + connect metal",
    "Frame template": None,
    "Probe device": "ppd_1port ΓÇö GSG pad reference",
    "Layer table": "Skyworks name ΓåÆ GDS (layer, datatype)",
}

rows = []
frame_size_str = "unknown size"

for label, path in input_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {label} ({path})")
    size = f"{path.stat().st_size:,} B"
    rows.append({"file": label, "path": path.name, "exists": True, "size": size, "role": input_roles[label]})

frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_lib.top_level()[0]
frame_bb = frame_cell.bounding_box()
if frame_bb is not None:
    (fx0, fy0), (fx1, fy1) = frame_bb
    frame_size_str = f"{fx1 - fx0:.1f}├ù{fy1 - fy0:.1f} ┬╡m"

for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

display(pd.DataFrame(rows))


## 2. Selection

**~30 s read**

Load the layermap, then identify which placed instances in the filter become R-tags.

| Sub-step | Module | Purpose |
|----------|--------|---------|
| 2.1 | `layermap.py` | Name ↔ GDS layer/datatype |
| 2.2 | `separate.py` | Resonator identification |

**Output:** `layermap`, `identification`, `res_df`, `res_list`, `parent`.

### 2.1 ΓÇö `layermap.py`

**~30 s read**

Loads `input_files/layermap` so later steps refer to layers by Skyworks names (`BAW_MBE`, `BAW_MTE`, ΓÇª) instead of raw GDS numbers.

**Call:** `load_layermap(LAYERMAP)` ΓåÆ `layermap` object with `.pair(name)` lookups.


In [ ]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

### 2.2 — `separate.py`

**~30 s read**

Finds placed resonators under the filter parent (`series*`, `shunt*`, `rcap*`, `mimcap*` masters). Returns dataframe rows plus live `Resonator` objects with placement transform preserved.

**Call:** `identify(FILTER)` → `identification` with `.resonators`, `.resonator_rows()`, `.parent`.

**Output:** `identification`, `res_df`, `res_list`, `parent` — one row/object per R-tag candidate (KB331: 8).

In [ ]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
res_list = identification.resonators
res_df = pd.DataFrame(identification.resonator_rows())  # one row per resonator

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}")
res_df

## 3. Separation

**~30 s read**

For each identified resonator, build an in-memory **PPD + resonator** assembly: GSG probe frame from `GSG_frame.gds` with the resonator centered and clearance-adjusted inside it. No die frame yet ΓÇö that is step 4.

**Output:** `ppd_assemblies` ΓÇö one per resonator, ready for frame placement.


### 3 ΓÇö PPD + orientation placement

**Logic:** center on PPD, then nudge to keep resonator metal **outside GSG pad keepout** (placement clearance) — not the step 5.2b collar-to-pad routing shift.

**~30 s read**

**Calls:** `prep_resonator_ppd` (with `identification` + `layermap`) ΓåÆ optional orientation analysis.

1. **Center** resonator bbox on the PPD template.
2. **Clearance nudge** ΓÇö ΓëÑ10 ┬╡m to GSG pad metal, ΓëÑ6 ┬╡m to release holes.
3. **Orientation shift** ΓÇö small search to maximize pad clearance while keeping DRC-friendly placement.

**Output:** `ppd_assemblies[index].top_cell` ΓÇö PPD ref + resonator ref in a scratch cell.


In [ ]:
from src.prep_resonator_ppd import assemblies_summary_df, prep_resonator_ppd

ppd_assemblies = prep_resonator_ppd(
    res_df,
    res_list,
    PPD,
    identification=identification,
    layermap=layermap,
)
print(f"Built {len(ppd_assemblies)} PPD + resonator assemblies")
display(assemblies_summary_df(ppd_assemblies))

## 4. Setting up

**~30 s read**

Place each PPD assembly into the **die frame template** (`KB331_N_Frame.gds`) and add the right-side MBE width filler. Margins are measured from the inner MBE ring cavity (not the outer 460├ù580 ┬╡m bbox).

- **X:** PPD/GSG frame left-aligned at 4% inner margin; wide resonators may get a **resonator-only** left shift (5 ┬╡m clearance to filler right edge).
- **Y:** assembly centered in 7% vertical margin band.
- **Filler:** MBE rectangle from inner-frame center line ΓåÆ right margin, full assembly height.

**Output:** `rteg_assemblies` ΓÇö frame + placed PPD/resonator + filler per index.


### 4 Die frame placement

**~30 s read**

**Call:** `prep_rteg_in_frame(ppd_assemblies, FRAME)` ΓåÆ `rteg_assemblies`

PPD and resonator are placed as **separate references** so only the resonator moves when enforcing filler clearance (`resonator_frame_shift` on the assembly object).

**4.1** copies filter `connectMTE` / `connectMBE` metal that touches each resonator into the frame cell (`attach_preserved_filter_interconnect_all`). GDS export is deferred to the final cell at the end of the notebook.


In [ ]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

### 4.1 — Preserved filter interconnect

Copy MTE/MBE polygons from the parent filter `connectMTE` / `connectMBE` cells that overlap each resonator (same rules as step 5.1), shifted into the RTEG frame coordinates, and written onto `assembly.top_cell` before export.

In [ ]:
from src.rteg_collect import (
    attach_preserved_filter_interconnect_all,
    preserved_interconnect_attach_rows,
)

preserved_by_index = attach_preserved_filter_interconnect_all(
    rteg_assemblies,
    res_list,
    identification,
    layermap,
)
inst_by_index = {i: r.inst_name for i, r in enumerate(res_list)}
preserved_attach_df = pd.DataFrame(
    preserved_interconnect_attach_rows(
        preserved_by_index,
        inst_names=inst_by_index,
    )
)
print(f"Attached preserved interconnect for {len(preserved_by_index)} assemblies")
display(preserved_attach_df)

## 5. Routing — overview

| Step | Module | What you get |
|------|--------|--------------|
| 5.1 | `rteg_collect` | Ground plates, preserved filter metal, release holes, body MTE/MBE |
| 5.2 | `rteg_classify` | Signal vs ground nodes; `mte_route_target` |
| 5.2b | `rteg_route_new` | If pad Y ∉ collar span: Y-align collar mouth to pad (+60 µm) for routing; re-prep (not pad clearance) |
| 5.2c | `rteg_mte_normalize` | `collar_extend` only: clip preserved MTE + MBE to body bbox + 20 µm; removes wild bus pieces that cause jagged signal routes and wild grounded caps |
| 5.3 | `rteg_route_new` | Signal route + ground filler, per resonator |
| 5.3b | `rteg_filler_keepout` | `collar_extend` only: carve rectangle filler outside MTE-extension span |
| 5.5 | `rteg_filler_bridge` | Bridge split rectangle plate + 5 µm frame-ring cap on the right |
| 5.4 | `rteg_route_clean` | Straighten BAW_REV keepout notches on MBE (2/0) |
| export | `export_route_new_gds` | One complete GDS per index (`route_outputs/`) |

**Routing split:** `center_pad` (**1, 3, 4, 6**) → MTE signal to center pad · `collar_extend` (**0, 2, 5, 7**) → MBE signal to center pad. The MBE ground filler is notched around the resonator and connected to ground either way.

### 5.1 ΓÇö Collect geometry roles

**~30 s read**

**Call:** `collect_geometry_roles(assembly, res, identification, layermap)` ΓåÆ `all_roles[index]`

Snapshots everything routing needs in **RTEG world coordinates**: GSG pad MBE, step-4 filler plate, preserved filter interconnect (MBE/MTE), resonator body metal, release holes, inner frame boundary.

Preserved MTE includes `connectMTE` tabs; series parts may also retain the stadium-adjacent collar (e.g. index 6 ΓåÆ areas 911 + 5191 ┬╡m┬▓).


In [ ]:
from src.rteg_collect import RtegCollectConfig, collect_geometry_roles

COLLECT_CONFIG = RtegCollectConfig()

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )

print(f"Collected geometry for {len(all_roles)} resonators")
display(pd.DataFrame(collect_rows).sort_values("index"))

### 5.2 — Classify GSG nodes

**~30 s read**

**Calls:** `collect_orientation_inputs` → `classify_nodes` → `all_classify[index]`

Labels top/center/bottom GSG nodes as signal or ground and sets **`mte_route_target`**:

- **`center_pad`** — preserved MTE faces the center pad → MTE carries signal (route to pad in 5.3).
- **`collar_extend`** — otherwise → MBE carries signal (route to pad in 5.3); the MTE extension stays grounded.

KB331: indices 1, 3, 4, 6 = `center_pad`; 0, 2, 5, 7 = `collar_extend`.

In [ ]:
from src.rteg_classify import classify_nodes
from src.rteg_collect import collect_orientation_inputs

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    orientation = collect_orientation_inputs(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        ground_plates=roles.ground_plates,
        config=COLLECT_CONFIG,
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        orientation=orientation,
        res_type=res.res_type,
    )
    all_classify[idx] = classification
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "mte_route_target": classification.mte_route_target,
            "signal_terminal": classification.signal_terminal,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
        }
    )

print(f"Classified {len(all_classify)} resonators")
display(pd.DataFrame(classify_overview_rows).sort_values("index"))

for idx, classification in all_classify.items():
    assert classification.mte_route_target in ("center_pad", "collar_extend")
    if classification.mte_route_target == "center_pad":
        assert classification.by_band()["center"].net == "signal"

print(f"Classification checks passed for all {len(all_classify)} resonators")

### 5.2b — Vertical signal shift (routing alignment, not pad clearance)

**Logic:** if pad Y falls outside the collar intercept span, shift the resonator vertically so collar mouth ≈ pad height (+60 µm same direction, GSG-clamped) for a cleaner horizontal signal route — then `shift_overrides` re-prep, re-attach interconnect, and re-collect before 5.3. (Step 3 already cleared metal off the pad.)


In [ ]:
from src.rteg_route_new import signal_shift_for_resonator

SIGNAL_SHIFT_EXTRA_UM = 60.0
SIGNAL_SHIFT_FRAME_MARGIN_UM = 10.0

signal_shift_rows: list[dict[str, object]] = []
shift_overrides: dict[int, tuple[float, float]] = {}

for idx, roles in all_roles.items():
    dx, dy = signal_shift_for_resonator(
        roles,
        all_classify[idx],
        layermap,
        extra_shift_um=SIGNAL_SHIFT_EXTRA_UM,
        frame_margin_um=SIGNAL_SHIFT_FRAME_MARGIN_UM,
    )
    asm = rteg_assemblies[idx]
    signal_shift_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "signal_shift_y": round(dy, 2),
            "applied": abs(dx) > 1e-6 or abs(dy) > 1e-6,
            "frame_shift_y_before": round(asm.resonator_frame_shift[1], 2),
        }
    )
    if abs(dx) > 1e-6 or abs(dy) > 1e-6:
        shift_overrides[idx] = (dx, dy)

display(pd.DataFrame(signal_shift_rows).sort_values("index"))
print(f"Non-zero shifts: {len(shift_overrides)}")

if shift_overrides:
    rteg_assemblies = prep_rteg_in_frame(
        ppd_assemblies, FRAME, shift_overrides=shift_overrides
    )
    attach_preserved_filter_interconnect_all(
        rteg_assemblies, res_list, identification, layermap,
    )

    all_roles = {}
    for idx in range(len(identification.resonators)):
        res = identification.resonators[idx]
        all_roles[idx] = collect_geometry_roles(
            rteg_assemblies[idx], res, identification, layermap, config=COLLECT_CONFIG,
        )

    all_classify = {}
    for idx, roles in all_roles.items():
        res = identification.resonators[idx]
        orientation = collect_orientation_inputs(
            rteg_assemblies[idx], res, identification, layermap,
            ground_plates=roles.ground_plates, config=COLLECT_CONFIG,
        )
        all_classify[idx] = classify_nodes(
            roles.ground_plates, roles.preserved,
            orientation=orientation, res_type=res.res_type,
        )
    print(f"Re-prepped and re-collected {len(all_roles)} resonators")


### 5.2c — Normalize preserved MTE + MBE for collar_extend resonators

**Logic — MTE:** For `collar_extend` resonators (indices 0, 2, 5, 7) the preserved filter `connectMTE` pieces can include the full interconnect bus with irregular wild edges far beyond the resonator collar. Step 5.3's `split_bridging_orphans` then produces a wild-shaped grounded MTE cap, causing the MBE rectangle filler to fail to connect.

**Logic — MBE:** The preserved `connectMBE` for these same resonators can be a single large filter-bus piece (4000–5000 µm²). Because its body-overlap fraction is low, `_split_collar_extensions` classifies it as an extension → `_find_upstream_knee` inserts a vertex from the wild polygon as an approach-angle knee → complex jagged signal route instead of a clean path to the pad.

Both layers are clipped to a bounding box around the resonator body + `collar_margin_um` (default 20 µm). After clipping the pieces are tightly overlapping the body → classified as collar (not extension) → `ext_polys` empty for `build_signal_route` → no knee constraint → clean direct route to the pad.

Updates `all_roles[idx].preserved.mte`, `all_roles[idx].preserved.mbe`, **and** the frame cell in-place.

In [ ]:
from src.rteg_mte_normalize import MteNormalizeConfig, normalize_mte_collar_extent_all

NORMALIZE_CFG = MteNormalizeConfig(collar_margin_um=20.0)

normalize_report = normalize_mte_collar_extent_all(
    all_roles,
    rteg_assemblies,
    all_classify,
    layermap,
    NORMALIZE_CFG,
)

print("Step 5.2c — MTE + MBE normalization (collar_extend only)")
for idx in sorted(normalize_report):
    if all_classify[idx].mte_route_target != "collar_extend":
        continue
    counts = normalize_report[idx]
    mte_areas = [round(tp.area_um2, 1) for tp in all_roles[idx].preserved.mte]
    mbe_areas = [round(tp.area_um2, 1) for tp in all_roles[idx].preserved.mbe]
    print(
        f"  [{idx}] {all_roles[idx].inst_name}: "
        f"removed MTE={counts['mte_removed']} MBE={counts['mbe_removed']} wild piece(s) → "
        f"MTE {len(mte_areas)} pieces {mte_areas}, MBE {len(mbe_areas)} pieces {mbe_areas}"
    )

### 5.3 — Routing (signal route + ground filler)

**~30 s read**

**Call:** `build_all_routes(all_roles, all_classify, layermap)` → `all_routes`

Reads the collar intercepts **directly from the die** `connectMTE`/`connectMBE` geometry (never computed), then for each resonator:

- **`center_pad` (1, 3, 4, 6):** MTE signal route from the center pad to the collar (preexisting extension merged in); MBE ground filler **traces the resonator body edge** (no interior overlap) and edge-touches the MBE ground.
- **`collar_extend` (0, 2, 5, 7):** MBE signal route to the center pad; the grounded **MTE extension is capped with MBE** so the filler connects from the top without overlapping the resonator. **Step 5.3b** (inside `build_all_routes`) then carves the rectangle filler **outside the horizontal MTE-extension intersection** so the plate does not wrap around the resonator — ground attaches only through the extension span (see `KB331_N_01_RTEG1_S1B.gds` for index 2).

Wild preserved MBE filter extensions and orphan MTE are removed after routing. Output: `all_routes[index]` with `signal_net`, `filler_nets`, and the die intercepts.

In [ ]:
from src.rteg_route_new import build_all_routes, routes_overview_rows

# Read intercepts from the die collar geometry; build signal routes + ground fillers.
# Step 5.3b (collar_extend only): carve rectangle filler outside MTE-extension span.
all_routes = build_all_routes(all_roles, all_classify, layermap)

routes_df = pd.DataFrame(routes_overview_rows(all_routes))
print(f"Routed {len(all_routes)} resonators (signal route + ground filler)")
display(routes_df)

### 5.4 — Clean MBE release-hole keepout curves (2/0)

**~15 s read**

**Call:** `clean_all_routes(all_routes, layermap, roles_by_index=all_roles)` → updated `all_routes`

Finds curved vertex runs on **BAW_MBE (2/0)** and straightens them to chords:

1. **Keepout-ring notches** — vertices on the **BAW_REV** keepout ring (release-hole radius + 6 µm clearance) from step 5.3 clearout.
2. **Smooth GDS arcs** — gently bowed runs (high interior angles, arc/chord ratio > 1.025) left by boolean merges; large detached filler joins are still ignored.

Each detected run is split into **at least three vertex clusters** and collapsed to straight segments.

In [ ]:
from src.rteg_route_clean import clean_all_routes, route_clean_overview_rows

# Preview keepout-notch stats (uses BAW_REV circles from step 5.1 roles).
clean_preview_df = pd.DataFrame(
    route_clean_overview_rows(all_routes, layermap, roles_by_index=all_roles)
)
display(clean_preview_df)

# Apply cleanup on MBE signal + filler polygons.
all_routes = clean_all_routes(all_routes, layermap, roles_by_index=all_roles)

routes_df = pd.DataFrame(routes_overview_rows(all_routes))
print(f"Cleaned release-hole keepout curves on {len(all_routes)} resonators")
display(routes_df)

### 5.5 — Bridge split MBE rectangle filler

**~10 s read**

**Call:** `apply_filler_bridge_all_routes(all_routes, all_roles, all_classify, layermap)` → updated `all_routes`

For **`collar_extend`** resonators where step 5.3b leaves the step-4 MBE width-filler rectangle as **two disconnected plate polygons** (KB331 index **0**, and index **7**):

1. Draw a **1 µm-wide** vertical strap on the rectangle **right edge** (GSG frame height on MBE 2/0) and boolean-merge filler nets into **one closed polygon** (spike-cleaned).
2. Append an **independent MBE cap** polygon on the same layer — same GSG height — from the filler right edge to **inner cavity right + 5 µm** (overlap into the die frame ring). The cap is **not** unioned into the filler plate.

No clearance rules apply to the bridge or cap.

In [ ]:
from src.rteg_filler_bridge import apply_filler_bridge_all_routes, filler_bridge_overview_rows

# Step 5.5: 1 µm right-edge bridge (top-right → bottom-right) when rectangle plate split in two.
bridge_preview = filler_bridge_overview_rows(
    all_routes, all_roles, all_classify, layermap,
)
if bridge_preview:
    display(pd.DataFrame(bridge_preview))
else:
    print("No split rectangle filler plates to bridge")

all_routes = apply_filler_bridge_all_routes(
    all_routes, all_roles, all_classify, layermap,
)

bridged = [row["index"] for row in bridge_preview]
if bridged:
    print(f"Bridged split rectangle filler at indices {bridged}")
    display(pd.DataFrame(routes_overview_rows(all_routes)).query("index in @bridged"))
else:
    print("Step 5.5 skipped — no qualifying split rectangle filler plates")

## Final export — complete routed RTEG

**~30 s read**

**Call:** `export_route_new_gds(rteg_assemblies, all_routes, ROUTE_OUTPUTS, layermap=..., parent=...)`

**Output:** `draft_output/route_outputs/{parent}_RTEG1_{index:02d}_{inst_name}.gds` — one file per resonator with die frame, resonator, the **signal route** (MTE/MBE → center pad) and the **ground filler** (notched to clear the resonator, connected to ground). Open each `.gds` with its sidecar `.lyp` in KLayout.

In [ ]:
# Final export — complete routed RTEG (signal route + ground filler) per resonator.

from src.export_gds import export_summary_df
from src.rteg_route_new import export_route_new_gds

ROUTE_OUTPUTS.mkdir(parents=True, exist_ok=True)
final_export_df = export_summary_df(
    export_route_new_gds(
        rteg_assemblies,
        all_routes,
        ROUTE_OUTPUTS,
        layermap=layermap,
        parent=parent,
    )
)

print(f"Exported {len(final_export_df)} routed RTEG GDS files to {ROUTE_OUTPUTS}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(final_export_df)